# ML Factory - Trading Ensemble Builder

Build production-ready ML trading models with a single notebook.

## Quick Start
1. **Run Cell 1** (Setup) - clones repo, installs dependencies
2. **Edit Cell 2** (Configuration) - toggle models, features, validation on/off
3. **Run remaining cells** - validates, loads data, trains, evaluates

## Features (All Toggleable in Cell 2)

| Category | Options |
|----------|---------|
| **Models** | XGBoost, LightGBM, CatBoost, Random Forest, SVM, LSTM, GRU, TCN, N-BEATS, InceptionTime, ResNet1D, PatchTST, iTransformer, TFT |
| **Features** | Price, Momentum, Volatility, Volume, Trend, Regime, Microstructure, Wavelet |
| **Feature Selection** | MDA, SHAP, Boruta, or use all features |
| **Validation** | Purged K-Fold, CPCV, PBO, Walk-Forward |
| **Ensembles** | Voting, Stacking, Blending with Ridge/MLP/XGBoost meta-learners |
| **Evaluation** | Backtest with configurable costs, Financial reports with charts |

## Requirements
- **GPU recommended** for neural/transformer models (Runtime > Change runtime type > GPU)
- **CPU works fine** for boosting models (XGBoost, LightGBM, CatBoost)

## Data
Default data files in `data/raw/`:
- MES (Micro E-mini S&P 500)
- MGC (Micro Gold)
- MCL (Micro Crude Oil)
- SI (Silver)
- NG (Natural Gas)

In [ ]:
# =============================================================
# CELL 1: SETUP - Run this once
# =============================================================
import os
import sys
import shutil

# Auto-detect environment: Colab vs Local
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    # ============== COLAB ENVIRONMENT ==============
    REPO_DIR = "/content/Research"
    
    # Force fresh clone to get latest data files
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    
    !git clone https://github.com/Snehpatel101/Research.git {REPO_DIR}
    
    # Install dependencies
    !cd {REPO_DIR} && pip install -q -r requirements-colab.txt 2>&1 | tail -1
    
    # Add to Python path
    sys.path.insert(0, REPO_DIR)
    
else:
    # ============== LOCAL ENVIRONMENT ==============
    # Assumes you're running from the repo directory or notebooks/ subdirectory
    REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath(".")))
    
    # Try to find the repo root
    for possible_root in [
        "/home/jake/Desktop/Research",
        os.path.dirname(os.path.abspath(".")),
        os.path.dirname(os.path.dirname(os.path.abspath("."))),
        os.getcwd(),
    ]:
        if os.path.exists(os.path.join(possible_root, "src", "factory.py")):
            REPO_DIR = possible_root
            break
    
    # Add to Python path if not already
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# Verify installation
try:
    from src.factory import MLFactory
    from src.config.experiment import ExperimentConfig
    print("ML Factory loaded successfully!")
except ImportError as e:
    print(f"ERROR: {e}")
    print("Check that the repository is accessible.")

# GPU check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU available: {gpu_name}")
else:
    print("No GPU detected. Boosting models work fine on CPU.")
    if IN_COLAB:
        print("For GPU: Runtime > Change runtime type > H100 GPU")

# Drive mount option (Colab web only)
DRIVE_MOUNTED = False
# Uncomment below for web Colab with Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_MOUNTED = True

print()
print("=" * 50)
print("Setup Complete")
print("=" * 50)
print(f"  Environment:   {'Google Colab' if IN_COLAB else 'Local'}")
print(f"  Repo:          {REPO_DIR}")
print(f"  Drive mounted: {DRIVE_MOUNTED}")
print()
print("AVAILABLE 1-MINUTE DATA FILES:")
print("  MES:  data/raw/mes-1m_data_2020.parquet")
print("  MGC:  data/raw/MGC-1m_data_2020.parquet")
print("  MCL:  data/raw/MCL-1m_data_2020.parquet")
print("  SI:   data/raw/si-1m_bk_2020.parquet")
print("  NG:   data/raw/ng-1m_bk_2020.parquet")
print()
print("Pipeline flow: 1m data -> resample to TARGET_TIMEFRAME -> MTF features")

In [ ]:
# =============================================================
# CELL 2: CONFIGURATION - Edit these settings
# =============================================================
# Toggle any option True/False or edit values. Defaults are sensible starting points.

# =============================================================
# SECTION 1: DATA SETTINGS
# =============================================================

SYMBOL = "MES"                          # Trading symbol
DATA_PATH = f"{REPO_DIR}/data/raw/mes-1m_data_2020.parquet"

# Alternative data paths (uncomment one):
# DATA_PATH = f"{REPO_DIR}/data/raw/MGC-1m_data_2020.parquet"  # Gold
# DATA_PATH = f"{REPO_DIR}/data/raw/MCL-1m_data_2020.parquet"  # Crude Oil
# DATA_PATH = f"{REPO_DIR}/data/raw/si-1m_bk_2020.parquet"     # Silver
# DATA_PATH = f"{REPO_DIR}/data/raw/ng-1m_bk_2020.parquet"     # Natural Gas

# Timeframe settings
INPUT_TIMEFRAME = "1min"                # Your raw data's timeframe
TARGET_TIMEFRAME = "5min"               # Resample to this for training

# Multi-timeframe features
MTF_ENABLED = True                      # Generate features from higher timeframes
MTF_TIMEFRAMES = ["15min", "30min", "1h"]

# =============================================================
# SECTION 2: MODEL SELECTION
# =============================================================

# ----- BOOSTING (Fast, CPU-friendly) -----
USE_XGBOOST = True
USE_LIGHTGBM = True
USE_CATBOOST = False

# ----- CLASSICAL ML (CPU only) -----
USE_RANDOM_FOREST = False
USE_LOGISTIC = False
USE_SVM = False                         # Slow on large datasets

# ----- NEURAL NETWORKS (GPU recommended) -----
USE_LSTM = False
USE_GRU = False
USE_TCN = False
USE_NBEATS = False

# ----- CNN-based (GPU recommended) -----
USE_INCEPTION_TIME = False
USE_RESNET_1D = False

# ----- TRANSFORMERS (GPU required, slow) -----
USE_PATCHTST = False
USE_ITRANSFORMER = False
USE_TFT = False

# =============================================================
# SECTION 3: FEATURE SETTINGS
# =============================================================

# Feature families to generate
USE_PRICE_FEATURES = True
USE_MOMENTUM_FEATURES = True
USE_VOLATILITY_FEATURES = True
USE_VOLUME_FEATURES = True
USE_TREND_FEATURES = True
USE_REGIME_FEATURES = False             # Market regime detection
USE_MICROSTRUCTURE_FEATURES = False     # Order flow features
USE_WAVELET_FEATURES = False            # Wavelet decomposition

# Feature selection
FEATURE_SELECTION_ENABLED = True        # Run feature importance selection
FEATURE_SELECTION_METHOD = "mda"        # "mda" (Mean Decrease Accuracy), "shap", "boruta"
FEATURE_SELECTION_N = 50                # Max features to keep (None = auto)

# Feature set presets (overrides family selection if set)
FEATURE_SET = None                      # None, "all", "boosting_optimal", "neural_optimal", "minimal"

# =============================================================
# SECTION 4: LABELING & TARGETS
# =============================================================

HORIZONS = [20]                         # Bars ahead to predict (list)
LABELING_METHOD = "triple_barrier"      # "triple_barrier", "fixed_horizon", "trend_scanning"

# Triple barrier settings (only used if LABELING_METHOD = "triple_barrier")
BARRIER_PROFIT_TAKE = 2.0               # ATR multiplier for take-profit
BARRIER_STOP_LOSS = 2.0                 # ATR multiplier for stop-loss
BARRIER_MAX_HOLDING = 50                # Max bars before timeout

# =============================================================
# SECTION 5: VALIDATION SETTINGS
# =============================================================

# Cross-validation method
CV_METHOD = "purged_kfold"              # "purged_kfold", "cpcv", "walk_forward"
CV_N_SPLITS = 5                         # Number of CV folds

# Advanced validation (slower but more rigorous)
RUN_CPCV = False                        # Combinatorial Purged Cross-Validation
RUN_PBO = False                         # Probability of Backtest Overfitting
RUN_WALK_FORWARD = False                # Walk-forward validation

# Leakage prevention
PURGE_BARS = 60                         # Bars to purge between train/test
EMBARGO_BARS = 10                       # Embargo period after test

# =============================================================
# SECTION 6: ENSEMBLE SETTINGS
# =============================================================

BUILD_ENSEMBLE = True                   # Combine models into ensemble
ENSEMBLE_METHOD = "stacking"            # "voting", "stacking", "blending"
META_LEARNER = "ridge_meta"             # "ridge_meta", "mlp_meta", "xgboost_meta"

# Ensemble diversity
ENSEMBLE_MIN_MODELS = 2                 # Minimum models for ensemble
ENSEMBLE_CALIBRATE = True               # Calibrate probabilities before ensemble

# =============================================================
# SECTION 7: HYPERPARAMETER TUNING
# =============================================================

OPTUNA_ENABLED = True                   # Run hyperparameter optimization
OPTUNA_TRIALS = 50                      # 25=quick, 50=balanced, 100+=production
OPTIMIZE_FOR = "sharpe_ratio"           # "sharpe_ratio", "f1_weighted", "accuracy", "sortino_ratio"

# =============================================================
# SECTION 8: BACKTESTING & EVALUATION
# =============================================================

RUN_BACKTEST = True                     # Run trading simulation
POSITION_SIZING = "fixed"               # "fixed", "kelly", "volatility", "confidence"
GENERATE_FINANCIAL_REPORT = True        # HTML/JSON financial report with charts

# Trading costs (for backtest)
COMMISSION_PER_TRADE = 2.50             # $ per trade
SLIPPAGE_TICKS = 1                      # Ticks of slippage
TICK_VALUE = 1.25                       # $ per tick (MES = 1.25)

# =============================================================
# SECTION 9: OUTPUT & LOGGING
# =============================================================

EXPERIMENT_NAME = "my_experiment"
RANDOM_SEED = 42
SAVE_RESULTS = True
VERBOSE = True                          # Detailed logging

# =============================================================
# BUILD CONFIGURATION (auto-generated from above)
# =============================================================

# Assemble model list from toggles
MODELS = []
if USE_XGBOOST: MODELS.append("xgboost")
if USE_LIGHTGBM: MODELS.append("lightgbm")
if USE_CATBOOST: MODELS.append("catboost")
if USE_RANDOM_FOREST: MODELS.append("random_forest")
if USE_LOGISTIC: MODELS.append("logistic")
if USE_SVM: MODELS.append("svm")
if USE_LSTM: MODELS.append("lstm")
if USE_GRU: MODELS.append("gru")
if USE_TCN: MODELS.append("tcn")
if USE_NBEATS: MODELS.append("nbeats")
if USE_INCEPTION_TIME: MODELS.append("inception_time")
if USE_RESNET_1D: MODELS.append("resnet_1d")
if USE_PATCHTST: MODELS.append("patchtst")
if USE_ITRANSFORMER: MODELS.append("itransformer")
if USE_TFT: MODELS.append("tft")

# Assemble feature families from toggles
FEATURE_FAMILIES = []
if USE_PRICE_FEATURES: FEATURE_FAMILIES.append("price")
if USE_MOMENTUM_FEATURES: FEATURE_FAMILIES.append("momentum")
if USE_VOLATILITY_FEATURES: FEATURE_FAMILIES.append("volatility")
if USE_VOLUME_FEATURES: FEATURE_FAMILIES.append("volume")
if USE_TREND_FEATURES: FEATURE_FAMILIES.append("trend")
if USE_REGIME_FEATURES: FEATURE_FAMILIES.append("regime")
if USE_MICROSTRUCTURE_FEATURES: FEATURE_FAMILIES.append("microstructure")
if USE_WAVELET_FEATURES: FEATURE_FAMILIES.append("wavelet")

# =============================================================
# CONFIGURATION SUMMARY
# =============================================================
print("=" * 60)
print("ML FACTORY CONFIGURATION")
print("=" * 60)

print("\n[DATA]")
print(f"  Symbol:           {SYMBOL}")
print(f"  Data path:        {DATA_PATH}")
print(f"  Target TF:        {TARGET_TIMEFRAME}")
print(f"  MTF enabled:      {MTF_ENABLED} {MTF_TIMEFRAMES if MTF_ENABLED else ''}")

print("\n[MODELS]")
print(f"  Selected:         {len(MODELS)} models")
for m in MODELS:
    print(f"    - {m}")

print("\n[FEATURES]")
print(f"  Families:         {FEATURE_FAMILIES}")
print(f"  Selection:        {FEATURE_SELECTION_METHOD if FEATURE_SELECTION_ENABLED else 'disabled'}")
print(f"  Feature set:      {FEATURE_SET or 'from families'}")

print("\n[VALIDATION]")
print(f"  CV method:        {CV_METHOD} ({CV_N_SPLITS} splits)")
print(f"  CPCV:             {RUN_CPCV}")
print(f"  PBO:              {RUN_PBO}")
print(f"  Walk-forward:     {RUN_WALK_FORWARD}")

print("\n[ENSEMBLE]")
print(f"  Build ensemble:   {BUILD_ENSEMBLE}")
print(f"  Method:           {ENSEMBLE_METHOD if BUILD_ENSEMBLE else 'N/A'}")
print(f"  Meta-learner:     {META_LEARNER if BUILD_ENSEMBLE else 'N/A'}")

print("\n[TRAINING]")
print(f"  Horizons:         {HORIZONS}")
print(f"  Labeling:         {LABELING_METHOD}")
print(f"  Optuna trials:    {OPTUNA_TRIALS if OPTUNA_ENABLED else 'disabled'}")
print(f"  Optimize for:     {OPTIMIZE_FOR}")

print("\n[EVALUATION]")
print(f"  Backtest:         {RUN_BACKTEST}")
print(f"  Financial report: {GENERATE_FINANCIAL_REPORT}")
print(f"  Position sizing:  {POSITION_SIZING}")

print("\n" + "=" * 60)

In [ ]:
# =============================================================
# CELL 3: VALIDATION - Run to check your configuration
# =============================================================
import os

# Valid options
VALID_MODELS = {
    "xgboost", "lightgbm", "catboost",
    "random_forest", "logistic", "svm",
    "lstm", "gru", "tcn", "nbeats",
    "inception_time", "resnet_1d",
    "patchtst", "itransformer", "tft",
}

NEURAL_MODELS = {
    "lstm", "gru", "tcn", "nbeats",
    "inception_time", "resnet_1d",
    "patchtst", "itransformer", "tft",
}

VALID_CV_METHODS = {"purged_kfold", "cpcv", "walk_forward"}
VALID_ENSEMBLE_METHODS = {"voting", "stacking", "blending"}
VALID_META_LEARNERS = {"ridge_meta", "mlp_meta", "xgboost_meta", "calibrated_meta"}
VALID_LABELING = {"triple_barrier", "fixed_horizon", "trend_scanning", "directional"}
VALID_FEATURE_SETS = {None, "all", "boosting_optimal", "neural_optimal", "minimal"}
VALID_OPTIMIZE_FOR = {"sharpe_ratio", "f1_weighted", "accuracy", "sortino_ratio", "profit_factor"}

errors = []
warnings = []

# --- Model validation ---
for m in MODELS:
    if m not in VALID_MODELS:
        errors.append(f"Unknown model: '{m}'. Valid: {sorted(VALID_MODELS)}")

if not MODELS:
    errors.append("No models selected. Enable at least one USE_* toggle.")

# --- Data validation ---
if not os.path.exists(DATA_PATH):
    warnings.append(f"DATA_PATH not found: {DATA_PATH}")

# --- Feature validation ---
if not FEATURE_FAMILIES and FEATURE_SET is None:
    errors.append("No features selected. Enable feature families or set FEATURE_SET.")

if FEATURE_SET not in VALID_FEATURE_SETS:
    errors.append(f"Invalid FEATURE_SET: '{FEATURE_SET}'. Valid: {VALID_FEATURE_SETS}")

# --- CV validation ---
if CV_METHOD not in VALID_CV_METHODS:
    errors.append(f"Invalid CV_METHOD: '{CV_METHOD}'. Valid: {VALID_CV_METHODS}")

# --- Ensemble validation ---
if BUILD_ENSEMBLE:
    if ENSEMBLE_METHOD not in VALID_ENSEMBLE_METHODS:
        errors.append(f"Invalid ENSEMBLE_METHOD: '{ENSEMBLE_METHOD}'. Valid: {VALID_ENSEMBLE_METHODS}")
    if META_LEARNER not in VALID_META_LEARNERS:
        errors.append(f"Invalid META_LEARNER: '{META_LEARNER}'. Valid: {VALID_META_LEARNERS}")
    if len(MODELS) < ENSEMBLE_MIN_MODELS:
        warnings.append(f"Ensemble needs {ENSEMBLE_MIN_MODELS}+ models, only {len(MODELS)} selected.")

# --- Labeling validation ---
if LABELING_METHOD not in VALID_LABELING:
    errors.append(f"Invalid LABELING_METHOD: '{LABELING_METHOD}'. Valid: {VALID_LABELING}")

# --- Optimization validation ---
if OPTIMIZE_FOR not in VALID_OPTIMIZE_FOR:
    errors.append(f"Invalid OPTIMIZE_FOR: '{OPTIMIZE_FOR}'. Valid: {VALID_OPTIMIZE_FOR}")

# --- GPU check for neural models ---
selected_neural = [m for m in MODELS if m in NEURAL_MODELS]
if selected_neural:
    import torch
    if not torch.cuda.is_available():
        warnings.append(
            f"GPU not available but neural models selected: {selected_neural}. "
            "Training will be VERY slow. Enable GPU: Runtime > Change runtime type > GPU"
        )

# --- Advanced validation warnings ---
if RUN_CPCV:
    warnings.append("CPCV enabled - this significantly increases training time.")
if RUN_PBO:
    warnings.append("PBO enabled - requires multiple CV runs, very slow.")
if RUN_WALK_FORWARD:
    warnings.append("Walk-forward enabled - adds validation time.")

# --- Print results ---
print("=" * 50)
print("CONFIGURATION VALIDATION")
print("=" * 50)

if errors:
    print("\nERRORS (must fix):")
    for e in errors:
        print(f"  [X] {e}")
    raise ValueError("Configuration has errors. Fix them in Cell 2 and re-run.")

if warnings:
    print("\nWARNINGS (optional to fix):")
    for w in warnings:
        print(f"  [!] {w}")

print("\n[OK] Configuration is valid!")
print(f"     {len(MODELS)} models, {len(FEATURE_FAMILIES)} feature families, {len(HORIZONS)} horizons")
print("=" * 50)

In [ ]:
# =============================================================
# CELL 4: LOAD & PREVIEW DATA
# =============================================================
import pandas as pd

# Load data based on file extension
if DATA_PATH.endswith(".parquet"):
    raw_data = pd.read_parquet(DATA_PATH)
elif DATA_PATH.endswith(".csv"):
    raw_data = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH}. Use .parquet or .csv")

# Normalize column names to lowercase
raw_data.columns = [c.lower().strip() for c in raw_data.columns]

# Validate required OHLCV columns
REQUIRED_COLUMNS = ["open", "high", "low", "close", "volume"]
missing = [c for c in REQUIRED_COLUMNS if c not in raw_data.columns]
if missing:
    raise ValueError(
        f"Missing required OHLCV columns: {missing}\n"
        f"Found columns: {list(raw_data.columns)}\n"
        f"The pipeline expects: {REQUIRED_COLUMNS}"
    )

# Ensure datetime index
if "datetime" in raw_data.columns:
    raw_data["datetime"] = pd.to_datetime(raw_data["datetime"])
    raw_data = raw_data.set_index("datetime").sort_index()
elif "date" in raw_data.columns:
    raw_data["date"] = pd.to_datetime(raw_data["date"])
    raw_data = raw_data.set_index("date").sort_index()
    raw_data.index.name = "datetime"
elif not isinstance(raw_data.index, pd.DatetimeIndex):
    # Try parsing the existing index as datetime
    try:
        raw_data.index = pd.to_datetime(raw_data.index)
        raw_data.index.name = "datetime"
        raw_data = raw_data.sort_index()
    except Exception:
        raise ValueError(
            "Could not find or parse a datetime column. "
            "Data must have a 'datetime' or 'date' column, or a datetime-parseable index."
        )

# --- Summary ---
print("=" * 50)
print("Data Loaded Successfully")
print("=" * 50)
print(f"  Symbol:      {SYMBOL}")
print(f"  Rows:        {len(raw_data):,}")
print(f"  Shape:       {raw_data.shape}")
print(f"  Columns:     {list(raw_data.columns)}")
print(f"  Date range:  {raw_data.index.min()} -> {raw_data.index.max()}")
print(f"  Index name:  {raw_data.index.name}")
print()

# Missing values
missing_counts = raw_data[REQUIRED_COLUMNS].isnull().sum()
total_missing = missing_counts.sum()
if total_missing > 0:
    print("WARNING: Missing values in OHLCV columns:")
    for col, count in missing_counts.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(raw_data)*100:.2f}%)")
else:
    print("No missing values in OHLCV columns.")
print()

# Preview
print("First 5 rows:")
display(raw_data.head())

print(f"\nData ready: 'raw_data' DataFrame with {len(raw_data):,} rows.")

In [ ]:
# =============================================================
# CELL 5: ASSEMBLE CONFIG & RUN FACTORY
# =============================================================
from src.config.experiment import (
    ExperimentConfig,
    DataSection,
    TrainingSection,
    EvaluationSection,
    ValidationSection,
)
from src.config.training import OptunaConfig
from src.config.data import FeatureConfig, LabelingConfig, MTFConfig
from src.config.cv import CVConfig
from src.factory import MLFactory

# --- Build Feature Config ---
feature_config = FeatureConfig(
    families=FEATURE_FAMILIES,
    feature_set=FEATURE_SET,
    selection_enabled=FEATURE_SELECTION_ENABLED,
    selection_method=FEATURE_SELECTION_METHOD if FEATURE_SELECTION_ENABLED else None,
    selection_n_features=FEATURE_SELECTION_N if FEATURE_SELECTION_ENABLED else None,
)

# --- Build Labeling Config ---
labeling_config = LabelingConfig(
    method=LABELING_METHOD,
    profit_take_atr=BARRIER_PROFIT_TAKE,
    stop_loss_atr=BARRIER_STOP_LOSS,
    max_holding_bars=BARRIER_MAX_HOLDING,
)

# --- Build MTF Config ---
mtf_config = MTFConfig(
    enabled=MTF_ENABLED,
    mode="indicators" if MTF_ENABLED else "none",
    timeframes=MTF_TIMEFRAMES if MTF_ENABLED else [],
    primary_timeframe=TARGET_TIMEFRAME,
)

# --- Build CV Config ---
cv_config = CVConfig(
    method=CV_METHOD,
    n_splits=CV_N_SPLITS,
    purge_bars=PURGE_BARS,
    embargo_bars=EMBARGO_BARS,
)

# --- Build Optuna Config ---
optuna_config = OptunaConfig(
    enabled=OPTUNA_ENABLED,
    n_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
    metric=OPTIMIZE_FOR,
)

# --- Assemble Full Config ---
config = ExperimentConfig(
    name=EXPERIMENT_NAME,
    random_seed=RANDOM_SEED,
    verbose=VERBOSE,
    
    data=DataSection(
        symbol=SYMBOL,
        data_path=DATA_PATH,
        features=feature_config,
        labeling=labeling_config,
        mtf=mtf_config,
    ),
    
    training=TrainingSection(
        models=MODELS,
        horizons=HORIZONS,
        build_ensemble=BUILD_ENSEMBLE,
        ensemble_method=ENSEMBLE_METHOD if BUILD_ENSEMBLE else None,
        meta_learner=META_LEARNER if BUILD_ENSEMBLE else None,
        ensemble_calibrate=ENSEMBLE_CALIBRATE if BUILD_ENSEMBLE else False,
        optuna=optuna_config,
        cv=cv_config,
    ),
    
    evaluation=EvaluationSection(
        run_backtest=RUN_BACKTEST,
        position_sizing=POSITION_SIZING,
        generate_financial_report=GENERATE_FINANCIAL_REPORT,
        commission_per_trade=COMMISSION_PER_TRADE,
        slippage_ticks=SLIPPAGE_TICKS,
        tick_value=TICK_VALUE,
    ),
    
    validation=ValidationSection(
        run_cpcv=RUN_CPCV,
        run_pbo=RUN_PBO,
        run_walk_forward=RUN_WALK_FORWARD,
    ),
)

# --- Print Configuration Summary ---
print("=" * 60)
print("EXPERIMENT CONFIGURATION ASSEMBLED")
print("=" * 60)
print(f"  Name:              {config.name}")
print(f"  Symbol:            {config.data.symbol}")
print(f"  Models:            {config.training.models}")
print(f"  Horizons:          {config.training.horizons}")
print(f"  Feature families:  {config.data.features.families}")
print(f"  Feature set:       {config.data.features.feature_set or 'from families'}")
print(f"  Feature selection: {config.data.features.selection_method or 'disabled'}")
print(f"  CV method:         {config.training.cv.method}")
print(f"  Ensemble:          {config.training.ensemble_method if config.training.build_ensemble else 'disabled'}")
print(f"  Optuna trials:     {config.training.optuna.n_trials if config.training.optuna.enabled else 'disabled'}")
print(f"  CPCV:              {config.validation.run_cpcv}")
print(f"  PBO:               {config.validation.run_pbo}")
print(f"  Walk-forward:      {config.validation.run_walk_forward}")
print(f"  Backtest:          {config.evaluation.run_backtest}")
print(f"  Financial report:  {config.evaluation.generate_financial_report}")
print("=" * 60)
print()

# --- Run the Factory ---
factory = MLFactory(config, enable_checkpoints=True)

try:
    print("Starting ML Factory pipeline...")
    print("This may take a while depending on model selection and settings.")
    print()
    result = factory.run()
    print()
    if result.success:
        print(result.summary())
    else:
        print(f"Pipeline completed with errors: {result.error_message}")
except KeyboardInterrupt:
    print("\n\nPipeline interrupted by user.")
    print("To resume from checkpoint, run:")
    print("  result = factory.resume_from_checkpoint()")
    result = None
except Exception as e:
    print()
    print(f"ERROR: Factory run failed: {e}")
    print()
    import traceback
    traceback.print_exc()
    print()
    print("To resume from the last checkpoint, run:")
    print("  result = factory.resume_from_checkpoint()")
    result = None

In [ ]:
# =============================================================
# CELL 6: RESULTS & VISUALIZATION
# =============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob

if result is None or not result.success:
    msg = "No successful result to display."
    if result and result.error_message:
        msg += f"\nError: {result.error_message}"
    print(msg)
else:
    print("=" * 60)
    print("EXPERIMENT RESULTS")
    print("=" * 60)
    print(f"  Run ID:          {result.run_id}")
    print(f"  Models trained:  {result.n_models}")
    print(f"  Best model:      {result.best_model}")
    print(f"  Duration:        {result.duration_seconds:.1f}s ({result.duration_seconds/60:.1f} min)")
    print()

    # --- Model Metrics Table ---
    if result.metrics:
        print("-" * 40)
        print("Model Performance")
        print("-" * 40)
        metrics_df = pd.DataFrame(result.metrics).T
        metrics_df.index.name = "model"
        display(metrics_df.round(4))
        print()

    # --- Ensemble Metrics ---
    if result.ensemble_metrics:
        print("-" * 40)
        print("Ensemble Metrics")
        print("-" * 40)
        for k, v in result.ensemble_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        print()

    # --- Backtest Metrics ---
    if result.backtest_metrics:
        print("-" * 40)
        print("Backtest Results")
        print("-" * 40)
        highlight_keys = ["sharpe_ratio", "max_drawdown", "profit_factor", "win_rate_pct"]
        for k in highlight_keys:
            if k in result.backtest_metrics:
                print(f"  {k}: {result.backtest_metrics[k]}")
        for k, v in result.backtest_metrics.items():
            if k not in highlight_keys:
                if isinstance(v, (int, float)):
                    print(f"  {k}: {v}")
        print()

    # --- Display Plot Images ---
    if result.output_dir and Path(result.output_dir).exists():
        plot_files = sorted(glob.glob(str(Path(result.output_dir) / "**" / "*.png"), recursive=True))[:6]
        if plot_files:
            print("-" * 40)
            print(f"Plots ({len(plot_files)} found)")
            print("-" * 40)
            n_plots = len(plot_files)
            cols = min(n_plots, 2)
            rows = (n_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
            if n_plots == 1:
                axes = [axes]
            else:
                axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, pf in enumerate(plot_files):
                img = mpimg.imread(pf)
                axes[i].imshow(img)
                axes[i].set_title(Path(pf).stem, fontsize=10)
                axes[i].axis("off")
            # Hide unused subplots
            for j in range(n_plots, len(axes)):
                axes[j].axis("off")
            plt.tight_layout()
            plt.show()

    if result.bundle_path:
        print(f"Bundle path: {result.bundle_path}")
    if result.output_dir:
        print(f"Output dir:  {result.output_dir}")

In [ ]:
# =============================================================
# CELL 7: SAVE RESULTS
# =============================================================
import shutil
from pathlib import Path

if not SAVE_RESULTS:
    print("SAVE_RESULTS is False. Skipping save.")
elif result is None or not result.success:
    print("No successful result to save.")
elif result.output_dir and Path(result.output_dir).exists():
    src_dir = Path(result.output_dir)

    # Determine save location based on environment and Drive mount
    if DRIVE_MOUNTED:
        save_dest = Path(f"/content/drive/MyDrive/ml_factory_results/{EXPERIMENT_NAME}")
        save_location = "Google Drive"
    else:
        save_dest = Path(f"{REPO_DIR}/experiments/results/{EXPERIMENT_NAME}")
        save_location = "Local" if not IN_COLAB else "Colab session"

    save_dest.mkdir(parents=True, exist_ok=True)

    print(f"Copying results...")
    print(f"  Source:      {src_dir}")
    print(f"  Destination: {save_dest}")
    print(f"  Location:    {save_location}")

    # Copy entire output directory
    if save_dest.exists():
        shutil.rmtree(save_dest)
    shutil.copytree(src_dir, save_dest)

    # Summary
    copied_files = list(save_dest.rglob("*"))
    n_files = sum(1 for f in copied_files if f.is_file())
    print()
    print("=" * 50)
    print("Save Complete")
    print("=" * 50)
    print(f"  Files copied:    {n_files}")
    print(f"  Location:        {save_location}")
    print(f"  Path:            {save_dest}")
    if result.bundle_path:
        try:
            bundle_in_dest = save_dest / Path(result.bundle_path).relative_to(src_dir)
            print(f"  Bundle:          {bundle_in_dest}")
        except ValueError:
            print(f"  Bundle:          {result.bundle_path}")
    print(f"  Experiment:      {EXPERIMENT_NAME}")

    if IN_COLAB and not DRIVE_MOUNTED:
        print()
        print("NOTE: Results in Colab session storage (will be lost on disconnect).")
        print("To download:")
        print(f"  !zip -r /content/results.zip {save_dest}")
        print("  from google.colab import files")
        print("  files.download('/content/results.zip')")
else:
    print("No output directory found in result. Nothing to save.")